# XGBoost с логарифмом цены

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
import joblib
import pickle

```
Создание колонки с логарифмом цены
```

In [3]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025.csv')

In [4]:
df['log_price'] = np.log(df['Цена'])

```
Преобразование категориальных признаков
```

In [5]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг',
               'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

```
Загрузка обучающей и тестовой выборок
```

In [7]:
X_train, X_test, y_train, y_test = joblib.load("data/data_split_log.pkl")

In [8]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=500,
        n_jobs=-1,
        random_state=42,
        eval_metric='rmse'
    ))
])

```
Обучение и сохранение модели
```

In [9]:
model.fit(X_train, y_train, regressor__verbose=True)
joblib.dump(model, "models/xgb_log_model.pkl")

['models/xgb_log_model.pkl']

```
Предсказание на тестовой выборке
```

In [10]:
y_pred_log = model.predict(X_test)

In [11]:
y_pred = np.exp(y_pred_log) # Обратное преобразование прогнозов
y_test_real = np.exp(y_test)

```
Оценка модели
```

In [12]:
xg_mse = mean_squared_error(y_test_real, y_pred)
xg_rmse = np.sqrt(xg_mse)
xg_mae = mean_absolute_error(y_test_real, y_pred)
xg_r2 = r2_score(y_test_real, y_pred)

```
Вывод результатов
```

In [13]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [xg_mse, xg_rmse, xg_mae, xg_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),99_383_589_560.28
1,Среднеквадратическая ошибка (RMSE),315_251.63
2,Средняя абсолютная ошибка (MAE),193_512.39
3,Коэффицент детерминации (R^2),0.90


```
Сохранение метрик в отдельный файл
```

In [14]:
metrics = {
    "model_name": "XGBoost Log",
    "MSE": xg_mse,
    "RMSE": xg_rmse,
    "MAE": xg_mae,
    "R2": xg_r2
}

In [15]:
with open("metrics/xgb_log_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)